## Credit Risk Project — Data Cleaning & Preparation

### Purpose of This Notebook
This notebook prepares the raw credit card default dataset for downstream modeling by:

- Validating encoded categorical variables
- Correcting known data quality issues
- Handling anomalous and out-of-domain values
- Ensuring features align with real-world credit risk logic

No feature engineering or modeling is performed in this phase.


### Importing Libraries

In [1]:
import pandas as pd
import numpy as np

### Loading the dataset

In [2]:
df=pd.read_excel("default of credit card clients.xls", header=1)
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


### Remove Non-Analytical Identifier

In [3]:
df= df.drop(columns=["ID"])

### Identifier Removal
The customer ID column is removed as it does not carry predictive information and should not be used in risk modeling.

### Validate Encoded Categorical Variables

In [4]:
df[['SEX', 'EDUCATION', 'MARRIAGE']].apply(pd.Series.value_counts)

,SEX,EDUCATION,MARRIAGE
0,NaN,14,54.0
1,11888.0,10585,13659.0
2,18112.0,14030,15964.0
3,NaN,4917,323.0
4,NaN,123,NaN
5,NaN,280,NaN
6,NaN,51,NaN


### Encoded Category Validation
Value counts reveal the presence of undocumented category codes in EDUCATION and MARRIAGE variables. These values must be consolidated into valid categories to preserve interpretability and prevent model distortion.


### Fix Invalid EDUCATION Values

In [5]:
df['EDUCATION']= df['EDUCATION'].replace([0,5,6], 4)
df['EDUCATION'].value_counts().sort_index()

,count
EDUCATION,
1,10585
2,14030
3,4917
4,468


### Fix Invalid MARRIAGE Values

In [6]:
df['MARRIAGE']=df['MARRIAGE'].replace(0, 3)
df['MARRIAGE'].value_counts().sort_index()

,count
MARRIAGE,
1,13659
2,15964
3,377


### Categorical Domain Correction
Undocumented category codes were identified in EDUCATION and MARRIAGE variables. These values were conservatively consolidated into valid "Other" categories to preserve interpretability, reduce noise, and ensure regulatory-safe feature behavior.

### REPAYMENT STATUS DOMAIN VALIDATION

In [9]:
repayment_cols=['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
df[repayment_cols].agg(['min', 'max'])

,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
min,-2,-2,-2,-2,-2,-2
max,8,8,8,8,8,8


### Repayment Status Validation — Results
All repayment status variables (PAY_0 to PAY_6) fall strictly within the expected domain of -2 to 8. This confirms that delinquency behavior is consistently encoded and free from out-of-range or corrupted values.

The presence of both negative and positive values indicates a realistic mix of non-users, timely payers, revolvers, and delinquent customers. These variables are therefore reliable primary predictors for credit default modeling.


### Exposure & Payment Consistency Analysis

In [14]:
payment_cols=[col for col in df.columns if 'PAY_AMT' in col]
bill_cols=[col for col in df.columns if 'BILL_AMT' in col]
df[payment_cols + bill_cols].describe()

,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6
count,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.000000,30000.000000,30000.000000
mean,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,51223.330900,49179.075167,4.701315e+04,43262.948967,40311.400967,38871.760400
std,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,73635.860576,71173.768783,6.934939e+04,64332.856134,60797.155770,59554.107537
min,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,-165580.000000,-69777.000000,-1.572640e+05,-170000.000000,-81334.000000,-339603.000000
25%,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,3558.750000,2984.750000,2.666250e+03,2326.750000,1763.000000,1256.000000
50%,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,22381.500000,21200.000000,2.008850e+04,19052.000000,18104.500000,17071.000000
75%,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,67091.000000,64006.250000,6.016475e+04,54506.000000,50190.500000,49198.250000
max,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,964511.000000,983931.000000,1.664089e+06,891586.000000,927171.000000,961664.000000


### Exposure vs Payment Behavior — Observations

Billing amounts (`BILL_AMT*`) exhibit a very wide distribution, with maximum balances exceeding 1.6M and negative minimum values. Negative billing balances likely reflect account credits, charge reversals, or overpayments applied to future statements. Such values are common in credit card portfolios and do not necessarily indicate data errors. However, their presence must be considered carefully during modeling to avoid unintended distortions.

Payment amounts (`PAY_AMT*`) also show substantial dispersion. Median payments are consistently lower than median billing balances, indicating that many customers revolve balances rather than paying in full. Zero payment values are present across all months, representing periods of non-payment and potential liquidity stress.

The presence of extreme high payments and balances suggests a highly heterogeneous customer base with varying credit usage intensity. These patterns confirm that both exposure and repayment capacity vary significantly and should be explicitly modeled in later stages.

Overall, payment and billing behavior align with real-world consumer credit portfolios and provide critical signals beyond repayment status alone.


### Exposure Concentration VS Risk Segmentation

In [15]:
df['LIMIT_BAL'].describe()
df['LIMIT_BAL'].quantile([0.5,0.75,0.9,0.95,0.99])

,LIMIT_BAL
0.50,140000.0
0.75,240000.0
0.90,360000.0
0.95,430000.0
0.99,500000.0


### Credit Exposure Concentration Analysis

The distribution of approved credit limits (`LIMIT_BAL`) shows a strong concentration of exposure among higher-limit customers.

Percentile analysis reveals:
- 50% of customers have credit limits ≤ 140,000
- 75% of customers have credit limits ≤ 240,000
- 90% of customers have credit limits ≤ 360,000
- 95% of customers have credit limits ≤ 430,000
- Top 1% of customers have credit limits up to 500,000

#### Business Interpretation
- A relatively small proportion of customers account for a disproportionately large share of total credit exposure
- Defaults among high-limit customers would result in materially higher losses compared to low-limit defaults
- Exposure concentration risk exists even if default rates are low in absolute terms

#### Risk Implications
- Credit limit should be treated as a key driver of loss severity (LGD proxy)
- High-limit segments require stricter monitoring and potentially differentiated risk thresholds
- Models must capture interaction effects between credit limit and repayment behavior


### Validate AGE

In [10]:
df['AGE'].describe()

,AGE
count,30000.000000
mean,35.485500
std,9.217904
min,21.000000
25%,28.000000
50%,34.000000
75%,41.000000
max,79.000000


In [16]:
df[df['AGE'] < 18].shape, df[df['AGE'] > 100].shape

((0, 24), (0, 24))

AGE values fall within reasonable human ranges, with no observations below legal adulthood or implausibly high ages. No age-based filtering is required.


### Missing Value Confirmation

In [12]:
df.isnull().sum().sum()

np.int64(0)

### Missing Data Assessment
No missing values are present in the dataset.

### Final Clean Dataset Snapshot

In [13]:
df.head()
df.shape

(30000, 24)

### Data Cleaning & Validation — Final Summary

This notebook completed the full data cleaning and validation phase for the credit card default dataset. All variables were reviewed for domain validity, internal consistency, and alignment with real-world consumer credit behavior.

Key outcomes:

Removed non-analytical identifier (ID)

Corrected undocumented categorical values in EDUCATION and MARRIAGE

Verified repayment status variables (PAY_0–PAY_6) fall within valid delinquency domains

Confirmed billing and payment variables reflect realistic credit behavior, including refunds and non-payment periods

Assessed credit limit concentration and identified exposure-driven risk implications

Validated age distribution and confirmed absence of missing values

The resulting dataset is clean, internally consistent, and modeling-ready, with no assumptions introduced that could distort downstream risk estimation.

Next Step:
Proceed to exploratory data analysis to quantify default behavior, understand feature distributions, and identify risk-segment patterns prior to feature engineering and model development.